# 01 · Config → Model: Build a Gemma 4 From Scratch

**Hardware**: 🟢 CPU, zero downloads. Every model here is randomly initialised from configs you write by hand.

## What you will do

1. Write all four Gemma 4 configs from scratch and assemble a working (tiny, nonsense-producing) model
2. Watch `audio_config=None` delete an entire modality
3. Read the KV-sharing and double-wide-MLP coupling off a live model — the finding from [index.md](../index.md) §5
4. Measure the PLE table and see why "E2B" means *effective* 2B
5. Round-trip through `save_pretrained` / `from_pretrained` and prove the logits are identical

The point is not the model — it is that **a `transformers` model is a pure function of its config**, and you can hold the whole thing in your head at this scale.

In [1]:
%pip install -q "transformers>=5.14" torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from transformers import (
    Gemma4Config, Gemma4TextConfig, Gemma4VisionConfig, Gemma4AudioConfig,
    Gemma4ForConditionalGeneration, AutoConfig, AutoModel,
)

torch.manual_seed(0)
print(torch.__version__)

2.13.0+cu130


## 1. Four configs

Everything is scaled down by roughly 10–20× from E2B so the model builds in under a second. The *structure* is identical — same field names, same derived behaviour.

Two constraints are worth knowing before you write your own:

- **`hidden_size_per_layer_input`** turns PLE on. Set it to `0` and the per-layer path disappears (that is what 31B and 26B-A4B do).
- **`num_kv_shared_layers` must leave at least one non-shared layer of *each* layer type.** The sharing boundary is at `num_hidden_layers - num_kv_shared_layers`; if no `full_attention` layer exists before it, the first shared global layer raises `KeyError: 'full_attention'` looking for a donor. Try it — it is a genuine architectural constraint, not a bug.

In [3]:
text_config = Gemma4TextConfig(
    vocab_size=1024,
    hidden_size=128,
    intermediate_size=256,
    num_hidden_layers=12,
    num_attention_heads=4,
    num_key_value_heads=1,        # MQA, like E2B
    head_dim=32,
    global_head_dim=64,           # global layers get bigger heads
    sliding_window=16,
    hidden_size_per_layer_input=16,   # PLE on
    vocab_size_per_layer_input=1024,
    num_kv_shared_layers=4,       # last 4 layers share KV
    use_double_wide_mlp=True,
)

vision_config = Gemma4VisionConfig(
    hidden_size=64, intermediate_size=128, num_hidden_layers=2,
    num_attention_heads=4, num_key_value_heads=4, head_dim=16,
    patch_size=16, pooling_kernel_size=3, position_embedding_size=256,
)

audio_config = Gemma4AudioConfig(
    hidden_size=64, num_hidden_layers=2, num_attention_heads=4, output_proj_dims=128,
)

config = Gemma4Config(
    text_config=text_config, vision_config=vision_config, audio_config=audio_config,
    image_token_id=1000, audio_token_id=1001, video_token_id=1002,
    boi_token_id=1003, eoi_token_id=1004, boa_token_id=1005, eoa_token_index=1006,
)

# __post_init__ already ran and derived the attention schedule:
for i, t in enumerate(text_config.layer_types):
    print(f"  layer {i:2d}: {t}")
print("\nrope_parameters:", text_config.rope_parameters)

  layer  0: sliding_attention
  layer  1: sliding_attention
  layer  2: sliding_attention
  layer  3: sliding_attention
  layer  4: sliding_attention
  layer  5: full_attention
  layer  6: sliding_attention
  layer  7: sliding_attention
  layer  8: sliding_attention
  layer  9: sliding_attention
  layer 10: sliding_attention
  layer 11: full_attention

rope_parameters: {'sliding_attention': {'rope_type': 'default', 'rope_theta': 10000.0}, 'full_attention': {'rope_type': 'proportional', 'partial_rotary_factor': 0.25, 'rope_theta': 1000000.0}}


Note what you did not have to specify: `layer_types` was generated (5 sliding : 1 global, last layer forced global), and each layer type got its own RoPE settings — θ=10,000 for sliding, θ=1,000,000 with `partial_rotary_factor=0.25` for global.

## 2. Config → model

In [4]:
model = Gemma4ForConditionalGeneration(config).eval()

total = sum(p.numel() for p in model.parameters())
print(f"total: {total/1e6:.2f}M parameters\n")
for name in ["model.vision_tower", "model.audio_tower", "model.language_model",
             "model.embed_vision", "model.embed_audio", "lm_head"]:
    n = sum(p.numel() for p in model.get_submodule(name).parameters())
    print(f"  {name:26s} {n/1e6:8.3f}M")

total: 3.01M parameters

  model.vision_tower            0.164M
  model.audio_tower             0.310M
  model.language_model          2.515M
  model.embed_vision            0.008M
  model.embed_audio             0.016M
  lm_head                       0.131M


In [5]:
# It runs. The output is nonsense (random weights) but the shapes are real.
ids = torch.randint(0, 1000, (1, 12))
with torch.no_grad():
    out = model(input_ids=ids)
print("logits:", tuple(out.logits.shape))   # [batch, seq, vocab]

logits: (1, 12, 1024)


## 3. Deleting a modality

The 31B and 26B-A4B checkpoints ship `"audio_config": null`. Here is what that one `null` does — no conditional logic anywhere in the modelling code, just a `None` check in `Gemma4Model.__init__`.

In [6]:
import copy

cfg_noaudio = copy.deepcopy(config)
cfg_noaudio.audio_config = None
m_noaudio = Gemma4ForConditionalGeneration(cfg_noaudio)

print("audio_tower :", m_noaudio.model.audio_tower)
print("embed_audio :", m_noaudio.model.embed_audio)
print("vision_tower:", type(m_noaudio.model.vision_tower).__name__)

saved = total - sum(p.numel() for p in m_noaudio.parameters())
print(f"\nparameters removed: {saved/1e6:.3f}M")

audio_tower : None
embed_audio : None
vision_tower: Gemma4VisionModel

parameters removed: 0.327M


## 4. The coupling that explains the design

[index.md](../index.md) §5 and [chapter 06](../../06-text-decoder/index.md) §5 claim that the layers which give up their KV projections are exactly the layers that get a double-wide MLP. Read it off the live model rather than taking anyone's word for it.

In [7]:
print(f"{'layer':>5} | {'type':<17} | {'kv_shared':<9} | {'mlp width':>9} | {'owns k_proj':<11} | donor")
print("-" * 72)
for i, layer in enumerate(model.model.language_model.layers):
    a = layer.self_attn
    owns_k = (not a.is_kv_shared_layer) and a.k_proj is not None
    print(f"{i:5d} | {a.layer_type:<17} | {str(a.is_kv_shared_layer):<9} | "
          f"{layer.mlp.intermediate_size:9d} | {str(owns_k):<11} | {a.store_full_length_kv}")

layer | type              | kv_shared | mlp width | owns k_proj | donor
------------------------------------------------------------------------
    0 | sliding_attention | False     |       256 | True        | False
    1 | sliding_attention | False     |       256 | True        | False
    2 | sliding_attention | False     |       256 | True        | False
    3 | sliding_attention | False     |       256 | True        | False
    4 | sliding_attention | False     |       256 | True        | False
    5 | full_attention    | False     |       256 | True        | True
    6 | sliding_attention | False     |       256 | True        | False
    7 | sliding_attention | False     |       256 | True        | True
    8 | sliding_attention | True      |       512 | False       | False
    9 | sliding_attention | True      |       512 | False       | False
   10 | sliding_attention | True      |       512 | False       | False
   11 | full_attention    | True      |       512 | False       |

Three things to read off that table:

1. The boundary is at `12 - 4 = 8`. Layers 8–11 are shared.
2. **MLP width doubles at exactly the same index.** `use_double_wide_mlp` is not a global flag — `Gemma4TextMLP.__init__` gates it on `is_kv_shared_layer`.
3. Exactly two layers have `store_full_length_kv=True` — the last non-shared layer *of each type*. Those are the donors. A sliding layer's keys are rotated with a different RoPE than a global layer's, so they are not interchangeable and you need one of each.

Now break it deliberately, to see the constraint enforce itself:

In [8]:
bad = copy.deepcopy(config)
bad.text_config.num_kv_shared_layers = 8   # boundary at 4; the only full_attention donor is at index 5
bad_model = Gemma4ForConditionalGeneration(bad).eval()
try:
    with torch.no_grad():
        bad_model(input_ids=ids)
except KeyError as e:
    print("KeyError:", e)
    print("\nNo unshared full_attention layer exists before the sharing boundary,")
    print("so shared_kv_states['full_attention'] is never populated.")

KeyError: 'full_attention'

No unshared full_attention layer exists before the sharing boundary,
so shared_kv_states['full_attention'] is never populated.


## 5. The PLE table

On E2B, `embed_tokens_per_layer` is 262144 × (35 × 256) ≈ 2.35B parameters — larger than the rest of the model. Only one row per token is ever read, which is what lets a ~5B-parameter checkpoint behave like an "effective 2B" model.

In [9]:
lm = model.model.language_model
print("embed_tokens          :", tuple(lm.embed_tokens.weight.shape))
print("embed_tokens_per_layer:", tuple(lm.embed_tokens_per_layer.weight.shape))
print(f"  = vocab_size_per_layer_input x (num_hidden_layers x hidden_size_per_layer_input)")
print(f"  = {text_config.vocab_size_per_layer_input} x ({text_config.num_hidden_layers} x {text_config.hidden_size_per_layer_input})")

ple = lm.embed_tokens_per_layer.weight.numel()
print(f"\nPLE table is {100*ple/total:.1f}% of this toy model's parameters")
print("(on E2B it is closer to 45%)")

# The packed row for one token, unpacked per layer:
per_layer = lm.get_per_layer_inputs(ids, None)
print("\nget_per_layer_inputs ->", tuple(per_layer.shape),
      "= [batch, seq, num_layers, ple_dim]")

embed_tokens          : (1024, 128)
embed_tokens_per_layer: (1024, 192)
  = vocab_size_per_layer_input x (num_hidden_layers x hidden_size_per_layer_input)
  = 1024 x (12 x 16)

PLE table is 6.5% of this toy model's parameters
(on E2B it is closer to 45%)



get_per_layer_inputs -> (1, 12, 12, 16) = [batch, seq, num_layers, ple_dim]


## 6. Round trip

A config plus a state dict is the whole model. Save it, load it, and prove nothing was lost.

In [10]:
import tempfile, os

with tempfile.TemporaryDirectory() as d:
    model.save_pretrained(d)
    print("files written:", sorted(os.listdir(d)))

    cfg2 = AutoConfig.from_pretrained(d)
    print("\nAutoConfig  ->", type(cfg2).__name__)
    print("  text_config  ->", type(cfg2.text_config).__name__)
    print("  vision_config->", type(cfg2.vision_config).__name__)
    print("  audio_config ->", type(cfg2.audio_config).__name__)

    model2 = Gemma4ForConditionalGeneration.from_pretrained(d).eval()
    with torch.no_grad():
        torch.testing.assert_close(model(input_ids=ids).logits, model2(input_ids=ids).logits)
    print("\nlogits identical after round trip ✓")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

files written: ['config.json', 'generation_config.json', 'model.safetensors']

AutoConfig  -> Gemma4Config
  text_config  -> Gemma4TextConfig
  vision_config-> Gemma4VisionConfig
  audio_config -> Gemma4AudioConfig


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]


logits identical after round trip ✓


`AutoConfig` rebuilt the three nested sub-config *classes* from plain JSON dicts. That is what `Gemma4Config.sub_configs` is for.

## 7. `AutoModel.from_config` is just a registry lookup

`Gemma4Model.__init__` never mentions `Gemma4VisionModel` by name. It passes the sub-config to `AutoModel.from_config`, which reads `model_type` and looks it up.

In [11]:
for cfg_ in (vision_config, audio_config, text_config):
    built = AutoModel.from_config(cfg_)
    print(f"{cfg_.model_type:16s} -> {type(built).__name__}")

gemma4_vision    -> Gemma4VisionModel
gemma4_audio     -> Gemma4AudioModel
gemma4_text      -> Gemma4TextModel


## 8. Compare against the real thing

Configs are small JSON files. You can read all four released sizes without downloading a single weight — this is the table in [index.md](../index.md) §5, reproduced live.

We read the raw JSON rather than going through `AutoConfig` here, deliberately. The released file is the ground truth and cannot be changed by a library upgrade; `transformers` 5.15 already treats some of these as *per-layer* attributes and refuses to serve them globally. When you want a fact about a checkpoint, read the checkpoint.

In [12]:
import json
from huggingface_hub import hf_hub_download

FIELDS = ["hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads",
          "sliding_window", "num_kv_shared_layers", "attention_k_eq_v",
          "hidden_size_per_layer_input", "use_double_wide_mlp",
          "use_bidirectional_attention", "enable_moe_block", "max_position_embeddings"]

sizes = ["E2B", "31B", "26B-A4B"]
raw = {}
for s in sizes:
    path = hf_hub_download(f"google/gemma-4-{s}-it", "config.json")
    raw[s] = json.load(open(path))

print(f"{'field':<30}" + "".join(f"{s:>14}" for s in sizes))
print("-" * (30 + 14 * len(sizes)))
for f in FIELDS:
    print(f"{f:<30}" + "".join(f"{str(raw[s]['text_config'].get(f, '-')):>14}" for s in sizes))

print(f"\n{'has audio_config':<30}" + "".join(f"{str(raw[s].get('audio_config') is not None):>14}" for s in sizes))
print(f"{'vision hidden_size':<30}" + "".join(f"{str(raw[s]['vision_config']['hidden_size']):>14}" for s in sizes))

print()
for s in sizes:
    lt = raw[s]["text_config"]["layer_types"]
    full = [i for i, x in enumerate(lt) if x == "full_attention"]
    print(f"{s:>10}: {len(lt)} layers, full_attention at {full}")

field                                    E2B           31B       26B-A4B
------------------------------------------------------------------------
hidden_size                             1536          5376          2816
num_hidden_layers                         35            60            30
num_attention_heads                        8            32            16
num_key_value_heads                        1            16             8
sliding_window                           512          1024          1024
num_kv_shared_layers                      20             0             0
attention_k_eq_v                       False          True          True
hidden_size_per_layer_input              256             0             0
use_double_wide_mlp                     True         False         False
use_bidirectional_attention             None        vision        vision
enable_moe_block                       False         False          True
max_position_embeddings               131072       

## Exercises

1. Set `hidden_size_per_layer_input=0` and rebuild. Which modules disappear from `Gemma4TextDecoderLayer`? Check with `print(model.model.language_model.layers[0])`.
2. Set `enable_moe_block=True` with `num_experts=8`, `top_k_experts=2`, `moe_intermediate_size=64`. Print a decoder layer — does the dense `mlp` disappear? (Chapter 07 explains the answer.)
3. Set `attention_k_eq_v=True`. Which layers lose `v_proj`, and why only those?
4. Find the largest `num_kv_shared_layers` that still builds and runs, and explain the number in terms of `layer_types`.
5. Compute E2B's PLE table size from its config alone, and express it as a fraction of the 10.25GB checkpoint in bf16.

## Where next

[Chapter 02](../../02-text-io/index.md) starts the actual pipeline: turning a `messages` list into `input_ids`.